# Template Web Scraping dengan Beautiful Soup

Web scraping adalah proses mengambil informasi terstruktur dari halaman web. Pada modul ini, `requests` digunakan untuk mengambil HTML, **Beautiful Soup** untuk membaca struktur HTML, dan `pandas` untuk menyimpan hasil sebagai tabel.

## Capaian pembelajaran

Setelah menyelesaikan modul ini, Anda dapat:

1. menjelaskan alur dasar web scraping;
2. mem-parsing HTML dengan `BeautifulSoup`;
3. memilih elemen menggunakan `find()`, `find_all()`, dan CSS selector;
4. membersihkan hasil menjadi data tabular; dan
5. melakukan scraping sederhana secara bertanggung jawab.

## 1. Alur kerja

```text
URL/HTML → unduh HTML → parse → pilih elemen → ekstrak → bersihkan → simpan
             requests    BeautifulSoup      Python       pandas/CSV
```

Beautiful Soup membaca HTML yang sudah tersedia; pustaka ini bukan browser dan tidak menjalankan JavaScript. Jika data baru muncul setelah JavaScript berjalan, cari API publik/resmi situs tersebut atau gunakan alat otomasi browser dengan izin yang sesuai.

## 2. Etika dan batasan

Sebelum mengambil data dari sebuah situs:

- baca ketentuan layanan dan kebijakan privasinya;
- periksa `/robots.txt` dan patuhi aturan yang berlaku;
- jangan mengambil data pribadi, berhak cipta, atau di balik autentikasi tanpa izin;
- gunakan identitas `User-Agent` yang jujur, `timeout`, jeda, dan jumlah permintaan yang kecil;
- hentikan proses saat server menolak akses (`403`, `429`, dan sejenisnya);
- utamakan API resmi bila tersedia.

`robots.txt` bukan izin hukum dan bukan pengganti ketentuan layanan. Keduanya perlu diperiksa.

## 3. Persiapan

Jalankan sel berikut satu kali. Di Jupyter, `%pip` memasang paket ke environment kernel yang sedang aktif.

In [1]:
%pip install -q beautifulsoup4 requests pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from importlib.metadata import version
from pathlib import Path
from urllib.parse import urljoin, urlsplit, urlunsplit
from urllib.robotparser import RobotFileParser

import pandas as pd
import requests
from bs4 import BeautifulSoup
from bs4.element import Tag

print("beautifulsoup4:", version("beautifulsoup4"))
print("requests      :", version("requests"))
print("pandas        :", version("pandas"))

beautifulsoup4: 4.15.0
requests      : 2.34.2
pandas        : 3.0.5


## 4. Latihan Implementasi dengan HTML lokal

Kita mulai dari HTML buatan sendiri. Keuntungannya: contoh selalu dapat dijalankan, tidak membebani situs lain, dan strukturnya mudah dipelajari. Perhatikan tag (`article`, `h2`, `p`, `a`), atribut `class`, dan atribut `href`.

In [3]:
HTML_CONTOH = """
<!doctype html>
<html lang="id">
  <head><title>Toko Buku Data</title></head>
  <body>
    <h1>Buku Pilihan</h1>
    <section id="katalog">
      <article class="buku unggulan" data-id="B001">
        <h2 class="judul">Dasar Data Mining</h2>
        <p class="harga">Rp125.000</p>
        <p class="stok">Tersedia</p>
        <a href="/buku/dasar-data-mining">Detail</a>
      </article>
      <article class="buku" data-id="B002">
        <h2 class="judul">Python untuk Analisis Data</h2>
        <p class="harga">Rp149.500</p>
        <p class="stok habis">Habis</p>
        <a href="/buku/python-analisis">Detail</a>
      </article>
      <article class="buku" data-id="B003">
        <h2 class="judul">Statistika Praktis</h2>
        <p class="harga">Rp98.000</p>
        <!-- Elemen stok sengaja tidak tersedia -->
        <a href="/buku/statistika-praktis">Detail</a>
      </article>
    </section>
  </body>
</html>
"""

## 5. Parsing dan menelusuri HTML

Parser `html.parser` tersedia di pustaka standar Python, sehingga tidak membutuhkan paket parser tambahan. `prettify()` membantu melihat struktur, tetapi tidak diperlukan saat ekstraksi.

In [4]:
soup = BeautifulSoup(HTML_CONTOH, "html.parser")
print(soup.prettify()[:700])

<!DOCTYPE html>
<html lang="id">
 <head>
  <title>
   Toko Buku Data
  </title>
 </head>
 <body>
  <h1>
   Buku Pilihan
  </h1>
  <section id="katalog">
   <article class="buku unggulan" data-id="B001">
    <h2 class="judul">
     Dasar Data Mining
    </h2>
    <p class="harga">
     Rp125.000
    </p>
    <p class="stok">
     Tersedia
    </p>
    <a href="/buku/dasar-data-mining">
     Detail
    </a>
   </article>
   <article class="buku" data-id="B002">
    <h2 class="judul">
     Python untuk Analisis Data
    </h2>
    <p class="harga">
     Rp149.500
    </p>
    <p class="stok habis">
     Habis
    </p>
    <a href="/buku/python-analisis">
     Detail
    </a>
   </article>
   <ar


### Metode pencarian penting

| Kebutuhan | Metode | Contoh |
|---|---|---|
| Elemen pertama | `find()` | `soup.find("h1")` |
| Semua elemen sejenis | `find_all()` | `soup.find_all("article")` |
| Elemen pertama dengan CSS selector | `select_one()` | `soup.select_one(".judul")` |
| Semua elemen dengan CSS selector | `select()` | `soup.select("article.buku")` |
| Teks bersih | `get_text(strip=True)` | `tag.get_text(strip=True)` |
| Nilai atribut secara aman | `get()` | `tag.get("href")` |

CSS selector ringkas: `#katalog` memilih `id`, `.buku` memilih `class`, `article.buku` menggabungkan tag dan class, sedangkan `article > h2` memilih anak langsung.

In [5]:
print("Judul halaman :", soup.title.get_text(strip=True))
print("Heading pertama:", soup.find("h1").get_text(strip=True))
print("Jumlah buku    :", len(soup.find_all("article", class_="buku")))
print("Buku unggulan :", soup.select_one("article.unggulan .judul").get_text(strip=True))
print("Semua judul    :", [tag.get_text(strip=True) for tag in soup.select(".buku .judul")])
print("Tautan pertama :", soup.select_one(".buku a").get("href"))

Judul halaman : Toko Buku Data
Heading pertama: Buku Pilihan
Jumlah buku    : 3
Buku unggulan : Dasar Data Mining
Semua judul    : ['Dasar Data Mining', 'Python untuk Analisis Data', 'Statistika Praktis']
Tautan pertama : /buku/dasar-data-mining


## 6. Ekstraksi dan pembersihan data

Halaman nyata sering memiliki nilai yang hilang. Karena itu, jangan langsung mengakses `.text` dari hasil `select_one()` tanpa memeriksa apakah elemennya ditemukan. Fungsi berikut mengubah harga menjadi bilangan bulat dan mengisi elemen yang hilang dengan `None`.

In [6]:
BASE_URL_CONTOH = "https://contoh.invalid"


def teks_atau_none(induk: Tag, selector: str) -> str | None:
    elemen = induk.select_one(selector)
    return elemen.get_text(" ", strip=True) if elemen else None


def harga_ke_int(teks_harga: str | None) -> int | None:
    if teks_harga is None:
        return None
    digit = "".join(karakter for karakter in teks_harga if karakter.isdigit())
    return int(digit) if digit else None


def ekstrak_buku(dokumen: BeautifulSoup) -> tuple[dict[str, object], ...]:
    def ekstrak_satu(kartu: Tag) -> dict[str, object]:
        tautan = kartu.select_one("a[href]")
        href = tautan.get("href") if tautan else None
        return {
            "id": kartu.get("data-id"),
            "judul": teks_atau_none(kartu, ".judul"),
            "harga_rupiah": harga_ke_int(teks_atau_none(kartu, ".harga")),
            "stok": teks_atau_none(kartu, ".stok"),
            "url": urljoin(BASE_URL_CONTOH, str(href)) if href else None,
        }

    return tuple(ekstrak_satu(kartu) for kartu in dokumen.select("article.buku"))

In [7]:
data_buku = ekstrak_buku(soup)
df_buku = pd.DataFrame(data_buku)

assert len(df_buku) == 3, "Jumlah baris tidak sesuai"
assert df_buku["id"].is_unique, "ID buku harus unik"
assert df_buku["harga_rupiah"].gt(0).all(), "Harga harus positif"

df_buku

,id,judul,harga_rupiah,stok,url
0,B001,Dasar Data Mining,125000,Tersedia,https://contoh.invalid/buku/dasar-data-mining
1,B002,Python untuk Analisis Data,149500,Habis,https://contoh.invalid/buku/python-analisis
2,B003,Statistika Praktis,98000,NaN,https://contoh.invalid/buku/statistika-praktis


### Menyimpan hasil

`index=False` mencegah nomor indeks DataFrame ikut menjadi kolom CSV. Encoding `utf-8` menjaga karakter non-ASCII tetap terbaca.

In [8]:
OUTPUT_PATH = Path("hasil_buku.csv")
df_buku.to_csv(OUTPUT_PATH, index=False, encoding="utf-8")
print(f"{len(df_buku)} baris disimpan ke {OUTPUT_PATH.resolve()}")

3 baris disimpan ke D:\RAYYAN\Kuliah\Semester 3\Penambangan Data\Materi\1 - Data Scraping & EDA\hasil_buku.csv


## 7. Contoh langsung pada situs latihan

Bagian ini memakai [Quotes to Scrape](https://quotes.toscrape.com/), sebuah situs latihan scraping. Fungsi di bawah hanya menerima HTTPS dan host tersebut, memeriksa `robots.txt`, membatasi waktu tunggu serta ukuran respons, dan menolak respons non-HTML.

Menurut RFC 9309, respons `4xx` pada `robots.txt` berarti berkas tersebut *unavailable* dan crawler boleh mengakses sumber daya; kegagalan jaringan atau `5xx` diperlakukan sebagai larangan (*fail closed*).

In [9]:
LIVE_URL = "https://quotes.toscrape.com/"
ALLOWED_LIVE_HOSTS = frozenset({"quotes.toscrape.com"})
USER_AGENT = "DM-PracticeBot/1.0 (educational exercise)"
REQUEST_TIMEOUT_SECONDS = 10
MAX_RESPONSE_BYTES = 1_000_000


def validasi_url_latihan(url: str) -> None:
    bagian = urlsplit(url)
    if bagian.scheme != "https" or bagian.hostname not in ALLOWED_LIVE_HOSTS:
        raise ValueError("URL harus memakai HTTPS dan host quotes.toscrape.com")


def diizinkan_robots(url: str) -> bool:
    bagian = urlsplit(url)
    robots_url = urlunsplit((bagian.scheme, bagian.netloc, "/robots.txt", "", ""))
    try:
        respons = requests.get(
            robots_url,
            headers={"User-Agent": USER_AGENT},
            timeout=REQUEST_TIMEOUT_SECONDS,
        )
        if 400 <= respons.status_code < 500:
            return True
        respons.raise_for_status()
    except requests.RequestException:
        return False

    parser = RobotFileParser()
    parser.set_url(robots_url)
    parser.parse(respons.text.splitlines())
    return parser.can_fetch(USER_AGENT, url)


def ambil_html_latihan(url: str) -> str:
    validasi_url_latihan(url)
    if not diizinkan_robots(url):
        raise PermissionError("Pengambilan data tidak diizinkan oleh robots.txt")

    respons = requests.get(
        url,
        headers={"User-Agent": USER_AGENT},
        timeout=REQUEST_TIMEOUT_SECONDS,
    )
    respons.raise_for_status()
    if "text/html" not in respons.headers.get("Content-Type", "").lower():
        raise ValueError("Respons bukan HTML")
    if len(respons.content) > MAX_RESPONSE_BYTES:
        raise ValueError("Respons melebihi batas ukuran")
    return respons.text

In [10]:
JALANKAN_SCRAPING_LANGSUNG = True

if JALANKAN_SCRAPING_LANGSUNG:
    try:
        html_live = ambil_html_latihan(LIVE_URL)
        soup_live = BeautifulSoup(html_live, "html.parser")
        data_kutipan = tuple(
            {
                "kutipan": teks_atau_none(kartu, ".text"),
                "penulis": teks_atau_none(kartu, ".author"),
            }
            for kartu in soup_live.select(".quote")
        )
        df_kutipan = pd.DataFrame(data_kutipan)
        display(df_kutipan.head(10))
    except (requests.RequestException, PermissionError, ValueError) as error:
        print(f"Scraping dihentikan dengan aman: {error}")
else:
    print("Ubah JALANKAN_SCRAPING_LANGSUNG menjadi True untuk mencoba contoh daring.")

,kutipan,penulis
0,“The world as we have created it is a process ...,Albert Einstein
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling
2,“There are only two ways to live your life. On...,Albert Einstein
3,"“The person, be it gentleman or lady, who has ...",Jane Austen
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe
5,“Try not to become a man of success. Rather be...,Albert Einstein
6,“It is better to be hated for what you are tha...,André Gide
7,"“I have not failed. I've just found 10,000 way...",Thomas A. Edison
8,“A woman is like a tea bag; you never know how...,Eleanor Roosevelt
9,"“A day without sunshine is like, you know, nig...",Steve Martin


## 8. Latihan mandiri

Gunakan `soup` dari HTML lokal untuk menjawab pertanyaan berikut.

1. Ambil hanya judul buku yang statusnya `Tersedia`.
2. Hitung rata-rata harga buku.
3. Ambil semua atribut `data-id` dengan satu CSS selector.
4. Ubah fungsi ekstraksi agar stok yang hilang menjadi string `Tidak diketahui`.

Coba kerjakan sebelum membuka solusi.

In [11]:


#1. Judul buku yang statusnya "Tersedia"
judul_tersedia = [
    teks_atau_none(kartu, ".judul")
    for kartu in soup.select("article.buku")
    if (teks_atau_none(kartu, ".stok") or "").lower() == "tersedia"
]
print("1. Judul buku tersedia :", judul_tersedia)

#2. Rata-rata harga buku
daftar_harga = [
    harga_ke_int(teks_atau_none(kartu, ".harga"))
    for kartu in soup.select("article.buku")
]
daftar_harga = [harga for harga in daftar_harga if harga is not None]
rata_rata_harga = sum(daftar_harga) / len(daftar_harga)
print(f"2. Rata-rata harga     : Rp{rata_rata_harga:,.2f}")
print("   (cek lewat pandas)  :", df_buku["harga_rupiah"].mean())

#3. Semua atribut data-id dengan satu CSS selector
daftar_id = [kartu["data-id"] for kartu in soup.select("article.buku[data-id]")]
print("3. Daftar data-id      :", daftar_id)

#4. Stok yang hilang menjadi "Tidak diketahui"
STOK_DEFAULT = "Tidak diketahui"


def ekstrak_buku_v2(dokumen: BeautifulSoup) -> tuple[dict[str, object], ...]:
    """Sama seperti ekstrak_buku(), tetapi stok kosong diisi STOK_DEFAULT."""

    def ekstrak_satu(kartu: Tag) -> dict[str, object]:
        tautan = kartu.select_one("a[href]")
        href = tautan.get("href") if tautan else None
        return {
            "id": kartu.get("data-id"),
            "judul": teks_atau_none(kartu, ".judul"),
            "harga_rupiah": harga_ke_int(teks_atau_none(kartu, ".harga")),
            "stok": teks_atau_none(kartu, ".stok") or STOK_DEFAULT,
            "url": urljoin(BASE_URL_CONTOH, str(href)) if href else None,
        }

    return tuple(ekstrak_satu(kartu) for kartu in dokumen.select("article.buku"))


df_buku_v2 = pd.DataFrame(ekstrak_buku_v2(soup))

assert df_buku_v2["stok"].notna().all(), "Tidak boleh ada stok kosong"
assert (df_buku_v2.loc[df_buku_v2["id"] == "B003", "stok"] == STOK_DEFAULT).all()

print("4. Hasil ekstraksi versi 2:")
df_buku_v2

1. Judul buku tersedia : ['Dasar Data Mining']
2. Rata-rata harga     : Rp124,166.67
   (cek lewat pandas)  : 124166.66666666667
3. Daftar data-id      : ['B001', 'B002', 'B003']
4. Hasil ekstraksi versi 2:


,id,judul,harga_rupiah,stok,url
0,B001,Dasar Data Mining,125000,Tersedia,https://contoh.invalid/buku/dasar-data-mining
1,B002,Python untuk Analisis Data,149500,Habis,https://contoh.invalid/buku/python-analisis
2,B003,Statistika Praktis,98000,Tidak diketahui,https://contoh.invalid/buku/statistika-praktis


## 9. Ringkasan

- `requests` mengambil respons HTTP; selalu gunakan `timeout` dan `raise_for_status()`.
- `BeautifulSoup(html, "html.parser")` membangun pohon HTML.
- `find()`/`find_all()` cocok untuk pencarian berbasis tag dan atribut; `select_one()`/`select()` cocok untuk CSS selector.
- Gunakan `get_text(strip=True)` untuk teks dan `.get()` untuk atribut opsional.
- Validasi jumlah baris, nilai kosong, tipe data, dan keunikan sebelum menyimpan data.
- Selector dapat rusak ketika desain situs berubah; simpan contoh HTML dan uji ekstraktor secara berkala.

## Referensi primer

- [Dokumentasi Beautiful Soup](https://beautiful-soup-4.readthedocs.io/en/latest/)
- [Beautiful Soup di PyPI](https://pypi.org/project/beautifulsoup4/)
- [Requests Quickstart](https://requests.readthedocs.io/en/stable/user/quickstart/)
- [Python `urllib.robotparser`](https://docs.python.org/3/library/urllib.robotparser.html)
- [RFC 9309 - Robots Exclusion Protocol](https://www.rfc-editor.org/rfc/rfc9309.html)